# 📚 City Library Management System

**Graded Mini Project — Part A**

A small, self-contained Python system for a community library that stores books,
tracks members, manages the borrow/return cycle, and generates operational reports.

## Problem Statement

A community library wants a small program to manage its daily operations. This
notebook implements a system that can **store books**, **track members**, and
**generate useful insights**, per the following requirements:

1. **Book Records** — Book ID, Title, Author, Genre, Availability; add books, update availability on issue/return.
2. **Member Records** — Member ID, Name, Age, Contact Info; a member may borrow multiple books, but only if available.
3. **Borrow & Return System** — issuing/returning updates availability and writes a transaction to a borrow log.
4. **Reports & Queries** — available books by genre, members who have borrowed, search by title/author, most popular genre.
5. **Functions & Logic** — the system is broken into small reusable functions/methods with clear guard conditions (e.g. can't issue an already-issued book).

## Notebook Roadmap

| Section | Contents |
|---|---|
| 1 | Design overview |
| 2 | Core data classes: `Book`, `Member` |
| 3 | `Library` engine: catalog, membership, borrow/return, reports |
| 4 | Sample data setup |
| 5 | Sample run — catalog & member listing |
| 6 | Sample run — borrow / return flow (incl. guard conditions) |
| 7 | Sample run — reports & search queries |
| 8 | Technical notes (design decisions & assumptions) |


## 1. Design Overview

The system is built with a small, readable **object-oriented design**:

- **`Book`** — a plain data holder for one book's fields, plus a availability flag.
- **`Member`** — a plain data holder for one member's fields.
- **`Library`** — the "engine" that owns three collections and exposes all
  reusable operations as methods:
  - `books` — `dict[book_id -> Book]` for O(1) lookup/update by ID.
  - `members` — `dict[member_id -> Member]` for O(1) lookup/update by ID.
  - `borrow_log` — a list of transaction records (`dict`s), one row per
    borrow/return event, newest last — this *is* the borrow log required by
    the spec, and it also powers the reports (popular genre, who has
    borrowed, etc.) without needing extra bookkeeping.

Every mutating action (add book, add member, issue, return) is its own
method with an explicit guard condition and a human-readable message, so the
control flow required by the spec ("prevent issuing a book if already
borrowed", etc.) is easy to find and to test.


In [2]:
# Standard-library only -- no external dependencies required.
from datetime import datetime
from collections import Counter


## 2. Core Data Classes

### `Book`

Stores the five fields required by the spec: Book ID, Title, Author, Genre,
and Availability (`"Available"` / `"Issued"`). New books always start out
`"Available"`.


In [3]:
class Book:
    """A single library book record."""

    def __init__(self, book_id, title, author, genre):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.genre = genre
        self.availability = "Available"  # "Available" or "Issued"

    def is_available(self):
        return self.availability == "Available"

    def __repr__(self):
        return (f"Book({self.book_id}, '{self.title}', {self.author}, "
                f"{self.genre}, {self.availability})")


### `Member`

Stores the four fields required by the spec: Member ID, Name, Age, and
Contact Info. A member's borrowing history is *not* duplicated here — it is
derived on demand from the `Library`'s central `borrow_log`, so there is a
single source of truth for who has what.


In [4]:
class Member:
    """A single library member record."""

    def __init__(self, member_id, name, age, contact):
        self.member_id = member_id
        self.name = name
        self.age = age
        self.contact = contact

    def __repr__(self):
        return f"Member({self.member_id}, '{self.name}', age={self.age}, {self.contact})"


## 3. The `Library` Engine

All the reusable operations the spec asks for live here as small, single-purpose
methods:

- **Catalog management:** `add_book`, `update_book_availability`
- **Membership management:** `add_member`
- **Borrow & return:** `issue_book`, `return_book` (each updates availability
  *and* appends a row to `borrow_log`)
- **Reports & queries:** `available_books_by_genre`, `members_who_borrowed`,
  `search_book`, `most_popular_genre`
- **Display helpers:** `display_all_books`, `display_all_members`, `display_borrow_log`

Guard conditions implemented:
- Can't add a book/member with a `book_id`/`member_id` that already exists.
- Can't issue a book that doesn't exist, is already issued, or a member ID
  that doesn't exist.
- Can't return a book that isn't currently issued, or that isn't checked out
  by the member requesting the return.


In [5]:
class Library:
    """The central engine tying books, members and transactions together."""

    def __init__(self, name="City Library"):
        self.name = name
        self.books = {}        # book_id -> Book
        self.members = {}      # member_id -> Member
        self.borrow_log = []   # list of transaction dicts

    # ------------------------------------------------------------------
    # 3.1  Book records
    # ------------------------------------------------------------------
    def add_book(self, book_id, title, author, genre):
        """Add a new book to the catalog. Returns (success, message)."""
        if book_id in self.books:
            return False, f"Book ID '{book_id}' already exists."
        self.books[book_id] = Book(book_id, title, author, genre)
        return True, f"Added book '{title}' ({book_id})."

    def update_book_availability(self, book_id, status):
        """Directly set a book's availability. Used internally by issue/return."""
        if book_id not in self.books:
            return False, f"Book ID '{book_id}' not found."
        if status not in ("Available", "Issued"):
            return False, "Status must be 'Available' or 'Issued'."
        self.books[book_id].availability = status
        return True, f"Book '{book_id}' marked {status}."

    # ------------------------------------------------------------------
    # 3.2  Member records
    # ------------------------------------------------------------------
    def add_member(self, member_id, name, age, contact):
        """Register a new member. Returns (success, message)."""
        if member_id in self.members:
            return False, f"Member ID '{member_id}' already exists."
        self.members[member_id] = Member(member_id, name, age, contact)
        return True, f"Added member '{name}' ({member_id})."

    # ------------------------------------------------------------------
    # 3.3  Borrow & return system
    # ------------------------------------------------------------------
    def issue_book(self, member_id, book_id):
        """A member borrows a book, subject to guard conditions."""
        if member_id not in self.members:
            return False, f"Member ID '{member_id}' not found."
        if book_id not in self.books:
            return False, f"Book ID '{book_id}' not found."

        book = self.books[book_id]
        if not book.is_available():
            return False, f"Cannot issue -- '{book.title}' is already Issued."

        # A member may hold multiple books; no cap is imposed by the spec,
        # so the only guard here is the book's own availability.
        book.availability = "Issued"
        self.borrow_log.append({
            "member_id": member_id,
            "book_id": book_id,
            "action": "BORROW",
            "timestamp": datetime.now(),
        })
        member = self.members[member_id]
        return True, f"'{book.title}' issued to {member.name}."

    def return_book(self, member_id, book_id):
        """A member returns a book, subject to guard conditions."""
        if member_id not in self.members:
            return False, f"Member ID '{member_id}' not found."
        if book_id not in self.books:
            return False, f"Book ID '{book_id}' not found."

        book = self.books[book_id]
        if book.is_available():
            return False, f"Cannot return -- '{book.title}' is not currently Issued."

        # Confirm this member is the one currently holding the book, by
        # checking the most recent BORROW record for this book.
        current_holder = self._current_holder(book_id)
        if current_holder is not None and current_holder != member_id:
            holder_name = self.members[current_holder].name
            return False, (f"Cannot return -- '{book.title}' was borrowed by "
                            f"{holder_name}, not this member.")

        book.availability = "Available"
        self.borrow_log.append({
            "member_id": member_id,
            "book_id": book_id,
            "action": "RETURN",
            "timestamp": datetime.now(),
        })
        member = self.members[member_id]
        return True, f"'{book.title}' returned by {member.name}."

    def _current_holder(self, book_id):
        """Look back through the log to find who currently holds a book."""
        for entry in reversed(self.borrow_log):
            if entry["book_id"] == book_id and entry["action"] in ("BORROW", "RETURN"):
                return entry["member_id"] if entry["action"] == "BORROW" else None
        return None

    # ------------------------------------------------------------------
    # 3.4  Reports & queries
    # ------------------------------------------------------------------
    def available_books_by_genre(self, genre):
        """All Available books in a given genre."""
        return [b for b in self.books.values()
                if b.genre.lower() == genre.lower() and b.is_available()]

    def members_who_borrowed(self):
        """Members who have at least one BORROW entry in the log,
        together with how many books they currently hold."""
        currently_out = Counter(
            e["member_id"] for e in self.borrow_log if e["action"] == "BORROW"
        )
        returned = Counter(
            e["member_id"] for e in self.borrow_log if e["action"] == "RETURN"
        )
        ever_borrowed = {e["member_id"] for e in self.borrow_log if e["action"] == "BORROW"}
        result = []
        for member_id in ever_borrowed:
            held_now = currently_out[member_id] - returned[member_id]
            result.append((self.members[member_id], held_now))
        return result

    def search_book(self, keyword):
        """Case-insensitive search across Title and Author."""
        keyword = keyword.lower()
        return [b for b in self.books.values()
                if keyword in b.title.lower() or keyword in b.author.lower()]

    def most_popular_genre(self):
        """The genre with the most ISSUE (borrow) events overall."""
        issued_genres = [self.books[e["book_id"]].genre
                          for e in self.borrow_log
                          if e["action"] == "BORROW" and e["book_id"] in self.books]
        if not issued_genres:
            return None, {}
        tally = Counter(issued_genres)
        top_genre, _ = tally.most_common(1)[0]
        return top_genre, dict(tally)

    # ------------------------------------------------------------------
    # 3.5  Display helpers
    # ------------------------------------------------------------------
    def display_all_books(self):
        header = f"{'ID':<6}{'Title':<28}{'Author':<20}{'Genre':<14}{'Status':<10}"
        print(header)
        print("-" * len(header))
        for b in self.books.values():
            print(f"{b.book_id:<6}{b.title:<28}{b.author:<20}{b.genre:<14}{b.availability:<10}")

    def display_all_members(self):
        header = f"{'ID':<6}{'Name':<18}{'Age':<6}{'Contact':<20}"
        print(header)
        print("-" * len(header))
        for m in self.members.values():
            print(f"{m.member_id:<6}{m.name:<18}{m.age:<6}{m.contact:<20}")

    def display_borrow_log(self):
        header = f"{'Time':<20}{'Action':<9}{'Member':<14}{'Book':<28}"
        print(header)
        print("-" * len(header))
        for e in self.borrow_log:
            member_name = self.members[e["member_id"]].name
            book_title = self.books[e["book_id"]].title
            ts = e["timestamp"].strftime("%Y-%m-%d %H:%M:%S")
            print(f"{ts:<20}{e['action']:<9}{member_name:<14}{book_title:<28}")


## 4. Sample Data Setup

Instantiate one `Library` and populate it with a small, varied catalog and
member list so every report has something meaningful to show.


In [6]:
library = Library("Downtown City Library")

# --- Seed books: (book_id, title, author, genre) ---------------------------
sample_books = [
    ("B01", "The Hobbit", "J.R.R. Tolkien", "Fantasy"),
    ("B02", "A Game of Thrones", "George R.R. Martin", "Fantasy"),
    ("B03", "Dune", "Frank Herbert", "Sci-Fi"),
    ("B04", "Foundation", "Isaac Asimov", "Sci-Fi"),
    ("B05", "The Silent Patient", "Alex Michaelides", "Mystery"),
    ("B06", "Gone Girl", "Gillian Flynn", "Mystery"),
    ("B07", "Sapiens", "Yuval Noah Harari", "Non-Fiction"),
    ("B08", "Educated", "Tara Westover", "Non-Fiction"),
    ("B09", "The Hobbit Companion", "Robert Foster", "Fantasy"),
]
for book_id, title, author, genre in sample_books:
    ok, msg = library.add_book(book_id, title, author, genre)
    print(msg)

print()

# --- Seed members: (member_id, name, age, contact) --------------------------
sample_members = [
    ("M01", "Aisha Khan", 29, "aisha.khan@email.com"),
    ("M02", "Ravi Kumar", 34, "ravi.kumar@email.com"),
    ("M03", "Lena Fischer", 22, "lena.fischer@email.com"),
]
for member_id, name, age, contact in sample_members:
    ok, msg = library.add_member(member_id, name, age, contact)
    print(msg)


Added book 'The Hobbit' (B01).
Added book 'A Game of Thrones' (B02).
Added book 'Dune' (B03).
Added book 'Foundation' (B04).
Added book 'The Silent Patient' (B05).
Added book 'Gone Girl' (B06).
Added book 'Sapiens' (B07).
Added book 'Educated' (B08).
Added book 'The Hobbit Companion' (B09).

Added member 'Aisha Khan' (M01).
Added member 'Ravi Kumar' (M02).
Added member 'Lena Fischer' (M03).


## 5. Sample Run — Catalog & Member Listing

A quick look at the full catalog and membership before any borrowing
happens: every book should show `Available`.


In [7]:
print("Full Book Catalog")
print("=" * 60)
library.display_all_books()


Full Book Catalog
ID    Title                       Author              Genre         Status    
------------------------------------------------------------------------------
B01   The Hobbit                  J.R.R. Tolkien      Fantasy       Available 
B02   A Game of Thrones           George R.R. Martin  Fantasy       Available 
B03   Dune                        Frank Herbert       Sci-Fi        Available 
B04   Foundation                  Isaac Asimov        Sci-Fi        Available 
B05   The Silent Patient          Alex Michaelides    Mystery       Available 
B06   Gone Girl                   Gillian Flynn       Mystery       Available 
B07   Sapiens                     Yuval Noah Harari   Non-Fiction   Available 
B08   Educated                    Tara Westover       Non-Fiction   Available 
B09   The Hobbit Companion        Robert Foster       Fantasy       Available 


In [8]:
print("Registered Members")
print("=" * 60)
library.display_all_members()


Registered Members
ID    Name              Age   Contact             
--------------------------------------------------
M01   Aisha Khan        29    aisha.khan@email.com
M02   Ravi Kumar        34    ravi.kumar@email.com
M03   Lena Fischer      22    lena.fischer@email.com


## 6. Sample Run — Borrow / Return Flow

This section demonstrates:

1. A normal borrow.
2. **The guard condition**: trying to issue a book that is already issued.
3. A normal return.
4. **The guard condition**: trying to return a book that isn't checked out.
5. The resulting borrow log.


In [9]:
print("-- Normal borrows --")
for member_id, book_id in [("M01", "B01"), ("M02", "B03"), ("M01", "B05"), ("M03", "B02")]:
    ok, msg = library.issue_book(member_id, book_id)
    print(f"[{'OK' if ok else 'DENIED'}] {msg}")


-- Normal borrows --
[OK] 'The Hobbit' issued to Aisha Khan.
[OK] 'Dune' issued to Ravi Kumar.
[OK] 'The Silent Patient' issued to Aisha Khan.
[OK] 'A Game of Thrones' issued to Lena Fischer.


In [10]:
print("-- Guard condition: issuing an already-issued book --")
ok, msg = library.issue_book("M03", "B01")   # B01 was just issued to M01
print(f"[{'OK' if ok else 'DENIED'}] {msg}")


-- Guard condition: issuing an already-issued book --
[DENIED] Cannot issue -- 'The Hobbit' is already Issued.


In [11]:
print("-- Normal return --")
ok, msg = library.return_book("M01", "B01")
print(f"[{'OK' if ok else 'DENIED'}] {msg}")

print()
print("-- Guard condition: returning a book that is not currently issued --")
ok, msg = library.return_book("M03", "B01")  # B01 was just returned above
print(f"[{'OK' if ok else 'DENIED'}] {msg}")

print()
print("-- Guard condition: wrong member returning someone else's book --")
ok, msg = library.return_book("M01", "B03")  # B03 is held by M02, not M01
print(f"[{'OK' if ok else 'DENIED'}] {msg}")


-- Normal return --
[OK] 'The Hobbit' returned by Aisha Khan.

-- Guard condition: returning a book that is not currently issued --
[DENIED] Cannot return -- 'The Hobbit' is not currently Issued.

-- Guard condition: wrong member returning someone else's book --
[DENIED] Cannot return -- 'Dune' was borrowed by Ravi Kumar, not this member.


In [12]:
print("Current Catalog Status After Transactions")
print("=" * 60)
library.display_all_books()


Current Catalog Status After Transactions
ID    Title                       Author              Genre         Status    
------------------------------------------------------------------------------
B01   The Hobbit                  J.R.R. Tolkien      Fantasy       Available 
B02   A Game of Thrones           George R.R. Martin  Fantasy       Issued    
B03   Dune                        Frank Herbert       Sci-Fi        Issued    
B04   Foundation                  Isaac Asimov        Sci-Fi        Available 
B05   The Silent Patient          Alex Michaelides    Mystery       Issued    
B06   Gone Girl                   Gillian Flynn       Mystery       Available 
B07   Sapiens                     Yuval Noah Harari   Non-Fiction   Available 
B08   Educated                    Tara Westover       Non-Fiction   Available 
B09   The Hobbit Companion        Robert Foster       Fantasy       Available 


In [13]:
print("Borrow Log")
print("=" * 60)
library.display_borrow_log()


Borrow Log
Time                Action   Member        Book                        
-----------------------------------------------------------------------
2026-09-12 06:17:51 BORROW   Aisha Khan    The Hobbit                  
2026-09-12 06:17:51 BORROW   Ravi Kumar    Dune                        
2026-09-12 06:17:51 BORROW   Aisha Khan    The Silent Patient          
2026-09-12 06:17:51 BORROW   Lena Fischer  A Game of Thrones           
2026-09-12 06:18:09 RETURN   Aisha Khan    The Hobbit                  


## 7. Sample Run — Reports & Queries

Demonstrating the four report/query requirements:

1. Available books in a given genre.
2. Members who have borrowed books (and how many they currently hold).
3. Search for a book by title or author.
4. The most popular genre based on issued books.


In [14]:
print("Available Fantasy books")
print("-" * 40)
for b in library.available_books_by_genre("Fantasy"):
    print(b)


Available Fantasy books
----------------------------------------
Book(B01, 'The Hobbit', J.R.R. Tolkien, Fantasy, Available)
Book(B09, 'The Hobbit Companion', Robert Foster, Fantasy, Available)


In [15]:
print("Members who have borrowed books")
print("-" * 40)
for member, held_now in library.members_who_borrowed():
    print(f"{member.name} ({member.member_id}) -- currently holding {held_now} book(s)")


Members who have borrowed books
----------------------------------------
Lena Fischer (M03) -- currently holding 1 book(s)
Aisha Khan (M01) -- currently holding 1 book(s)
Ravi Kumar (M02) -- currently holding 1 book(s)


In [16]:
print("Search results for 'hobbit' (matches Title or Author)")
print("-" * 40)
for b in library.search_book("hobbit"):
    print(b)

print()
print("Search results for 'Herbert' (matches Title or Author)")
print("-" * 40)
for b in library.search_book("Herbert"):
    print(b)


Search results for 'hobbit' (matches Title or Author)
----------------------------------------
Book(B01, 'The Hobbit', J.R.R. Tolkien, Fantasy, Available)
Book(B09, 'The Hobbit Companion', Robert Foster, Fantasy, Available)

Search results for 'Herbert' (matches Title or Author)
----------------------------------------
Book(B03, 'Dune', Frank Herbert, Sci-Fi, Issued)


In [17]:
top_genre, tally = library.most_popular_genre()
print("Genre issue counts:", tally)
print(f"Most popular genre so far: {top_genre}")


Genre issue counts: {'Fantasy': 2, 'Sci-Fi': 1, 'Mystery': 1}
Most popular genre so far: Fantasy


### One more borrow, to show the popular-genre report update

Let's issue one more Sci-Fi book (`Foundation`) and re-run the report to
confirm the tally responds correctly to new transactions -- Sci-Fi should
now be tied with Fantasy at the top.


In [18]:
ok, msg = library.issue_book("M02", "B04")
print(f"[{'OK' if ok else 'DENIED'}] {msg}")

print()
top_genre, tally = library.most_popular_genre()
print("Genre issue counts:", tally)
print(f"Most popular genre now: {top_genre}  (ties are broken by first-reached-that-count)")


[OK] 'Foundation' issued to Ravi Kumar.

Genre issue counts: {'Fantasy': 2, 'Sci-Fi': 2, 'Mystery': 1}
Most popular genre now: Fantasy  (ties are broken by first-reached-that-count)


## 8. Technical Notes (Design Decisions & Assumptions)

**Design.** The system uses three small classes: `Book` and `Member` as plain
data holders, and `Library` as the engine exposing every required operation
as a single-purpose method (`add_book`, `add_member`, `issue_book`,
`return_book`, `search_book`, `available_books_by_genre`,
`members_who_borrowed`, `most_popular_genre`). Books and members are stored
in dictionaries keyed by ID for O(1) lookup and update, which keeps the
borrow/return logic simple and fast even as the catalog grows. All mutating
methods return an `(ok, message)` tuple rather than raising exceptions or
printing directly, so the same logic can drive a notebook demo, a CLI, or a
future GUI/test-suite without change.

**Borrow log as single source of truth.** Rather than storing "current
borrower" on the `Book` and a separate "borrowed books" list on `Member`
(which can drift out of sync), the system keeps one append-only
`borrow_log` of `BORROW`/`RETURN` events. Current holder, borrowing history,
and the most-popular-genre report are all derived from this log on demand.

**Guard conditions implemented.** Duplicate Book/Member IDs are rejected;
issuing checks the book exists, the member exists, and the book is currently
`Available`; returning checks the book is currently `Issued` and that the
member requesting the return is the actual current holder (prevents a
different member from "returning" someone else's book).

**Assumptions.** No limit is placed on how many books one member may hold
simultaneously (the spec does not require a cap). "Most popular genre" is
computed from cumulative issue events in the log, not from currently-issued
books only, so it reflects overall demand over time. Search matching is
case-insensitive substring matching on Title and Author. Ages and contact
info are stored as given, with no format validation, since the spec does not
require it. Persistence (saving to disk/DB) is out of scope for Part A; all
data lives in memory for the duration of the notebook session.
